In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/Real-Time-Sign-Language-Recognition.zip" -d "/content/drive/MyDrive/Real-Time-Sign-Language-Recognition_Project"
print("Đã giải nén xong!")

Đã giải nén xong!


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Điều hướng tới thư mục chứa project vừa giải nén trên Drive
%cd /content/drive/MyDrive/Real-Time-Sign-Language-Recognition_Project

# Liệt kê các file hiện có để kiểm tra xem đã vào đúng thư mục gốc chưa
print("\n--- DANH SÁCH FILE TRONG THƯ MỤC HIỆN TẠI ---")
!ls

# Đảm bảo thư mục lưu trữ tồn tại
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Kiểm tra GPU (Phải hiện ra thông tin của T4 GPU)
print("\n--- TRẠNG THÁI GPU ---")
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

/content/drive/MyDrive/Real-Time-Sign-Language-Recognition_Project

--- DANH SÁCH FILE TRONG THƯ MỤC HIỆN TẠI ---
models	Real-Time-Sign-Language-Recognition  results

--- TRẠNG THÁI GPU ---
GPU Available:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
%cd Real-Time-Sign-Language-Recognition
!ls

/content/drive/MyDrive/Real-Time-Sign-Language-Recognition_Project/Real-Time-Sign-Language-Recognition
configs  models     README.md  requirements.txt  src
data	 notebooks  reports    results


In [ ]:
import sys
sys.path.append('src/models')

from model_config import INPUT_SHAPE, NUM_CLASSES
from architectures import build_lstm_model, build_bilstm_model, build_cnn1d_model

print(f"Cấu hình từ T03: INPUT_SHAPE={INPUT_SHAPE}, NUM_CLASSES={NUM_CLASSES}")

Cấu hình từ T03: INPUT_SHAPE=(30, 534), NUM_CLASSES=30


In [ ]:
data_dir = "data/holistic/"
X_train = np.load(f"{data_dir}X_train_aug.npy")
y_train = np.load(f"{data_dir}y_train_aug.npy")
X_val = np.load(f"{data_dir}X_val.npy")
y_val = np.load(f"{data_dir}y_val.npy")
X_test = np.load(f"{data_dir}X_test.npy")
y_test = np.load(f"{data_dir}y_test.npy")

print("\n--- KIỂM TRA SHAPE DỮ LIỆU ---")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

# Đảm bảo nhãn đang ở dạng mảng 1 chiều (Label Encoding)
assert len(y_train.shape) == 1, "LỖI: Nhãn không phải Label Encoding 1 chiều!"
print("\n-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!")


--- KIỂM TRA SHAPE DỮ LIỆU ---
X_train shape: (3000, 30, 534)
y_train shape: (3000,)
X_val shape: (210, 30, 534)
y_val shape: (210,)

-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!


In [ ]:
# CẤU HÌNH HUẤN LUYỆN CHUNG
EPOCHS = 50
BATCH_SIZE = 32

In [ ]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN LSTM ---")
lstm_model = build_lstm_model()

lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

lstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('models/lstm_model.h5', monitor='val_loss', save_best_only=True)
]

lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=lstm_callbacks,
    verbose=1
)

# Lưu lịch sử train để sau này vẽ biểu đồ
with open('results/lstm_history.json', 'w') as f:
    json.dump(lstm_history.history, f)

print(f"\n[*] LSTM training completed.")
print(f"[*] Final validation accuracy: {lstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: models/lstm_model.h5")


--- XÂY DỰNG & HUẤN LUYỆN LSTM ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1634 - loss: 3.0463

94/94 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - accuracy: 0.2640 - loss: 2.6183 - val_accuracy: 0.5048 - val_loss: 1.7701
Epoch 2/50
90/94 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5117 - loss: 1.6035

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5580 - loss: 1.4495 - val_accuracy: 0.5381 - val_loss: 1.4562
Epoch 3/50
92/94 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6727 - loss: 1.0757

...

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8637 - loss: 0.4573 - val_accuracy: 0.7571 - val_loss: 1.0180
Epoch 6/50
92/94 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8550 - loss: 0.4889

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8203 - loss: 0.6075 - val_accuracy: 0.7333 - val_loss: 0.9414
Epoch 7/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8626 - loss: 0.4217

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8653 - loss: 0.4170 - val_accuracy: 0.7571 - val_loss: 0.8404
Epoch 8/50
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9023 - loss: 0.3109

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9107 - loss: 0.2970 - val_accuracy: 0.7667 - val_loss: 0.8051
Epoch 9/50
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9164 - loss: 0.2867

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9090 - loss: 0.3070 - val_accuracy: 0.8190 - val_loss: 0.6597
Epoch 10/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9223 - loss: 0.2512 - val_accuracy: 0.7476 - val_loss: 1.0067
Epoch 11/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8300 - loss: 0.6485 - val_accuracy: 0.7619 - val_loss: 0.9827
Epoch 12/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9233 - loss: 0.2429 - val_accuracy: 0.8333 - val_loss: 0.7120
Epoch 13/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9423 - loss: 0.2013 - val_accuracy: 0.8143 - val_loss: 0.8790
Epoch 14/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9077 - loss: 0.3404 - val_accuracy: 0.7952 - val_loss: 0.9456
Epoch 15/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9553 - loss: 0.1420 - val_accuracy: 0.7952 - val_loss: 0.9073
Epoch 16/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9550 - loss: 0.1629 - val_accuracy: 0.8333 - val_loss

In [ ]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN BiLSTM ---")
bilstm_model = build_bilstm_model()

bilstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

bilstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('models/bilstm_model.h5', monitor='val_loss', save_best_only=True)
]

bilstm_history = bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=bilstm_callbacks,
    verbose=1
)

with open('results/bilstm_history.json', 'w') as f:
    json.dump(bilstm_history.history, f)

print(f"\n[*] BiLSTM training completed.")
print(f"[*] Final validation accuracy: {bilstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: models/bilstm_model.h5")


--- XÂY DỰNG & HUẤN LUYỆN BiLSTM ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0768 - loss: 5.3610

94/94 ━━━━━━━━━━━━━━━━━━━━ 16s 79ms/step - accuracy: 0.0623 - loss: 7.0257 - val_accuracy: 0.0476 - val_loss: 3.6695
Epoch 2/50
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0492 - loss: 3.4885

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.0553 - loss: 3.3905 - val_accuracy: 0.0667 - val_loss: 3.2967
Epoch 3/50
92/94 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0856 - loss: 3.2317

...

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5123 - loss: 1.4754 - val_accuracy: 0.5143 - val_loss: 1.6207
Epoch 10/50
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6052 - loss: 1.2392

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5933 - loss: 1.2599 - val_accuracy: 0.5571 - val_loss: 1.4895
Epoch 11/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6370 - loss: 1.0720 - val_accuracy: 0.4762 - val_loss: 1.6688
Epoch 12/50
92/94 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6124 - loss: 1.1965

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6357 - loss: 1.1095 - val_accuracy: 0.5619 - val_loss: 1.3951
Epoch 13/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6483 - loss: 1.0881 - val_accuracy: 0.5714 - val_loss: 1.4440
Epoch 14/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6823 - loss: 0.9626 - val_accuracy: 0.6095 - val_loss: 1.5433
Epoch 15/50
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7617 - loss: 0.7091

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7727 - loss: 0.6936 - val_accuracy: 0.6333 - val_loss: 1.3459
Epoch 16/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7887 - loss: 0.6211 - val_accuracy: 0.6286 - val_loss: 1.3829
Epoch 17/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8193 - loss: 0.5490 - val_accuracy: 0.6429 - val_loss: 1.4596
Epoch 18/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8096 - loss: 0.5282

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8107 - loss: 0.5538 - val_accuracy: 0.6476 - val_loss: 1.3209
Epoch 19/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8497 - loss: 0.4603 - val_accuracy: 0.6476 - val_loss: 1.4417
Epoch 20/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8800 - loss: 0.3731 - val_accuracy: 0.6905 - val_loss: 1.3966
Epoch 21/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8610 - loss: 0.3959 - val_accuracy: 0.6667 - val_loss: 1.4878
Epoch 22/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8310 - loss: 0.5253 - val_accuracy: 0.6714 - val_loss: 1.4979
Epoch 23/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8403 - loss: 0.4740 - val_accuracy: 0.6476 - val_loss: 1.4365
Epoch 24/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8667 - loss: 0.4103 - val_accuracy: 0.6571 - val_loss: 1.4866
Epoch 25/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9030 - loss: 0.3031 - val_accuracy: 0.6476 - val_l

In [ ]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN ---")
cnn1d_model = build_cnn1d_model()

cnn1d_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cnn1d_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('models/cnn1d_model.h5', monitor='val_loss', save_best_only=True)
]

cnn1d_history = cnn1d_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn1d_callbacks,
    verbose=1
)

with open('results/cnn1d_history.json', 'w') as f:
    json.dump(cnn1d_history.history, f)

print(f"\n[*] 1D-CNN training completed.")
print(f"[*] Final validation accuracy: {cnn1d_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: models/cnn1d_model.h5")


--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1925 - loss: 2.8563

94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.3203 - loss: 2.3175 - val_accuracy: 0.4714 - val_loss: 1.6322
Epoch 2/50
84/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6000 - loss: 1.3024

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6643 - loss: 1.1137 - val_accuracy: 0.6524 - val_loss: 1.0204
Epoch 3/50
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7791 - loss: 0.7364

...

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9183 - loss: 0.2762 - val_accuracy: 0.8476 - val_loss: 0.5533
Epoch 7/50
84/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9310 - loss: 0.2497

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9327 - loss: 0.2335 - val_accuracy: 0.8571 - val_loss: 0.4394
Epoch 8/50
87/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9368 - loss: 0.2124

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9440 - loss: 0.1884 - val_accuracy: 0.8905 - val_loss: 0.4195
Epoch 9/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9557 - loss: 0.1600 - val_accuracy: 0.8619 - val_loss: 0.4895
Epoch 10/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9617 - loss: 0.1353 - val_accuracy: 0.8476 - val_loss: 0.5144
Epoch 11/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9710 - loss: 0.1162 - val_accuracy: 0.8667 - val_loss: 0.5409
Epoch 12/50
85/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9578 - loss: 0.1416

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9567 - loss: 0.1459 - val_accuracy: 0.8952 - val_loss: 0.4163
Epoch 13/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9820 - loss: 0.0749 - val_accuracy: 0.8714 - val_loss: 0.4347
Epoch 14/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9673 - loss: 0.1189 - val_accuracy: 0.8524 - val_loss: 0.6172
Epoch 15/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9600 - loss: 0.1153 - val_accuracy: 0.8619 - val_loss: 0.5842
Epoch 16/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9790 - loss: 0.0745 - val_accuracy: 0.8762 - val_loss: 0.4948
Epoch 17/50
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9653 - loss: 0.1023

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9680 - loss: 0.0965 - val_accuracy: 0.8952 - val_loss: 0.3899
Epoch 18/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9883 - loss: 0.0432 - val_accuracy: 0.8667 - val_loss: 0.6192
Epoch 19/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9807 - loss: 0.0620 - val_accuracy: 0.8667 - val_loss: 0.5322
Epoch 20/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9870 - loss: 0.0509 - val_accuracy: 0.9000 - val_loss: 0.4371
Epoch 21/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9883 - loss: 0.0354 - val_accuracy: 0.8667 - val_loss: 0.6043
Epoch 22/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9927 - loss: 0.0276 - val_accuracy: 0.9143 - val_loss: 0.4585
Epoch 23/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9967 - loss: 0.0172 - val_accuracy: 0.9143 - val_loss: 0.4421
Epoch 24/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9933 - loss: 0.0282 - val_accuracy: 0.8857 - val_loss: 0.5

In [ ]:
from tensorflow.keras.models import load_model

print("\n--- KIỂM TRA MODEL ĐĐÃ LƯU ---")
saved_models = {
    "LSTM": "models/lstm_model.h5",
    "BiLSTM": "models/bilstm_model.h5",
    "1D-CNN": "models/cnn1d_model.h5"
}

sample = X_test[:1]

for name, path in saved_models.items():
    if os.path.exists(path):
        # Tải mô hình lên
        test_model = load_model(path)

        # Cho mô hình dự đoán thử 1 mẫu
        pred = test_model.predict(sample, verbose=0)

        # Kiểm tra xem đầu ra có đúng là 30 class không
        assert pred.shape == (1, NUM_CLASSES), f"Lỗi shape ở {name}"

        print(f"-> [OK] {name} tải thành công. Output Shape: {pred.shape}")
    else:
        print(f"-> [LỖI] Không tìm thấy file {path}")


--- KIỂM TRA MODEL ĐĐÃ LƯU ---


-> [OK] LSTM tải thành công. Output Shape: (1, 30)


-> [OK] BiLSTM tải thành công. Output Shape: (1, 30)


-> [OK] 1D-CNN tải thành công. Output Shape: (1, 30)


In [ ]:
!zip -r models_and_results.zip models/ results/
print("Đã nén xong!")

updating: models/ (stored 0%)
updating: models/.gitkeep (stored 0%)
updating: models/lstm_model.h5 (deflated 6%)
updating: models/bilstm_model.h5 (deflated 9%)
updating: models/cnn1d_model.h5 (deflated 20%)
updating: results/ (stored 0%)
updating: results/.gitkeep (stored 0%)
updating: results/lstm_history.json (deflated 54%)
updating: results/bilstm_history.json (deflated 55%)
updating: results/cnn1d_history.json (deflated 56%)
Đã nén xong!
